# Diabetic Hospital Readmission — Data Cleaning Pipeline

**Dataset:** `fairlearn.datasets.fetch_diabetes_hospital`  
**Goal:** Predict early readmission (`<30` days) — binary classification.  
**Pipeline Steps:**
1. Load & Inspect
2. Fix Target Variable
3. Drop Leakage & High-Missing Columns
4. Replace String `'Missing'` with `NaN`
5. Impute Missing Values
6. Encode Categoricals
7. Final Validation

In [ ]:
# ============================================================
# Cell 1 — Imports
# ============================================================
import sys
import numpy as np
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder
from fairlearn.datasets import fetch_diabetes_hospital

print("Python:", sys.executable)
print(f"pandas {pd.__version__} | numpy {np.__version__}")

In [ ]:
# ============================================================
# Cell 2 — Load & Inspect
# ============================================================
def load_dataset() -> pd.DataFrame:
    """
    Loads the fairlearn diabetes hospital dataset and returns
    a single DataFrame with all features including the raw
    'readmitted' column (needed for target construction).
    """
    data = fetch_diabetes_hospital(as_frame=True)
    df = data.data.copy()
    return df


df = load_dataset()

print("Shape          :", df.shape)
print("Columns        :", df.columns.tolist())
print("\nDtypes:\n", df.dtypes.value_counts())
print("\nFirst 3 rows:")
df.head(3)

In [ ]:
# ============================================================
# Cell 3 — Fix Target Variable
# ============================================================
# BUG FIX: data.target from fairlearn is already numeric (readmit_binary),
# NOT the string 'readmitted' column.  Using data.target directly with
# x == '<30' would ALWAYS return 0 — silently breaking the target.
#
# Correct approach: build the binary label from the 'readmitted' string
# column that is already present in data.data.
#   <30  → 1  (early readmission — high risk)
#   >30 / NO → 0

def make_target(df: pd.DataFrame, col: str = 'readmitted') -> pd.Series:
    """
    Creates a binary target: 1 if readmitted within 30 days, else 0.

    Parameters
    ----------
    df  : source DataFrame (must contain the raw readmitted column)
    col : name of the raw readmission column (default 'readmitted')

    Returns
    -------
    pd.Series of dtype int (0 or 1)
    """
    if col not in df.columns:
        raise KeyError(f"Column '{col}' not found. Available: {df.columns.tolist()}")
    return (df[col] == '<30').astype(int)


df['target'] = make_target(df)

print("Target distribution:")
print(df['target'].value_counts())
print(f"\nPositive rate: {df['target'].mean():.2%}")

# Sanity check — must have BOTH classes present
assert df['target'].nunique() == 2, \
    "ERROR: target column only has one class — check 'readmitted' values!"
assert df['target'].isin([0, 1]).all(), \
    "ERROR: unexpected values in target column."
print("\n✅ Target check passed.")

In [ ]:
# ============================================================
# Cell 4 — Drop Leakage & High-Missing Columns
# ============================================================
# Columns to drop and why:
#   encounter_id   — unique row identifier, no predictive value
#   patient_nbr    — unique patient identifier, no predictive value
#   weight         — ~97% missing in original dataset
#   payer_code     — ~40% missing and low predictive relevance
#   medical_specialty — ~49% missing (confirmed in EDA)
#   readmitted     — raw source of our target (data leakage)
#   readmit_binary — direct derivation of target (data leakage)

COLS_TO_DROP = [
    'encounter_id',
    'patient_nbr',
    'weight',
    'payer_code',
    'medical_specialty',
    'readmitted',       # leakage — source of target
    'readmit_binary',   # leakage — direct derivation of target
]

before = df.shape[1]
df.drop(columns=[c for c in COLS_TO_DROP if c in df.columns], inplace=True)
after = df.shape[1]

print(f"Dropped {before - after} columns. New shape: {df.shape}")
print("Remaining columns:", df.columns.tolist())

In [ ]:
# ============================================================
# Cell 5 — Replace String 'Missing' with NaN
# ============================================================
# The dataset encodes absent values as the string 'Missing'.
# We unify them with proper NaN so pandas imputation works correctly.

def replace_string_missing(df: pd.DataFrame, sentinel: str = 'Missing') -> pd.DataFrame:
    """
    Replaces a sentinel string (e.g. 'Missing') with np.nan across all columns.
    Uses vectorized pd.DataFrame.replace — no Python-level loops.
    """
    return df.replace(sentinel, np.nan)


df = replace_string_missing(df)

# Report missing counts
missing = df.isnull().sum()
missing_nonzero = missing[missing > 0].sort_values(ascending=False)
if missing_nonzero.empty:
    print("No missing values found after replacement.")
else:
    print("Columns with NaN after replacement:")
    print(missing_nonzero)

In [ ]:
# ============================================================
# Cell 6 — Impute Missing Values
# ============================================================
# Strategy:
#   Categorical / string columns → fill with 'Unknown'
#   Numeric columns              → fill with column median
#
# Pandas 2+/4+ compliance:
#   - Use include='str' instead of deprecated include='object'
#   - Avoid chained inplace assignment (copy-on-write safe)

def impute_missing(df: pd.DataFrame) -> pd.DataFrame:
    """
    Imputes missing values in-place (copy-on-write compatible).

    - String / categorical columns: fills NaN with 'Unknown'.
    - Numeric columns: fills NaN with the column median.

    Returns the imputed DataFrame.
    """
    df = df.copy()  # avoid mutating caller's reference

    # --- Categorical columns ---
    try:
        # pandas 3+: use 'str' dtype selector
        str_cols = df.select_dtypes(include='str').columns
    except Exception:
        # fallback for pandas < 3
        str_cols = df.select_dtypes(include='object').columns

    df[str_cols] = df[str_cols].fillna('Unknown')

    # --- Numeric columns (excluding the target) ---
    num_cols = df.select_dtypes(include='number').columns.difference(['target'])
    df[num_cols] = df[num_cols].fillna(df[num_cols].median())

    return df


df = impute_missing(df)

remaining_nulls = df.isnull().sum().sum()
print(f"Missing values after imputation: {remaining_nulls}")
assert remaining_nulls == 0, "ERROR: nulls still present after imputation!"
print("✅ Imputation check passed.")

In [ ]:
# ============================================================
# Cell 7 — Encode Categorical Columns
# ============================================================
# Uses sklearn OrdinalEncoder instead of a LabelEncoder loop.
# Advantages:
#   - Handles all categorical columns in one fitted object
#   - Encoder can be saved and reused for inference
#   - Inverse-transform is supported per-column

def encode_categoricals(
    df: pd.DataFrame,
    exclude_cols: list = None
) -> tuple[pd.DataFrame, OrdinalEncoder, list]:
    """
    Ordinal-encodes all string/object columns (excluding `exclude_cols`).

    Parameters
    ----------
    df           : DataFrame (already imputed — no NaNs in cat. cols)
    exclude_cols : columns to skip (e.g. already-numeric targets)

    Returns
    -------
    df_encoded   : DataFrame with encoded columns
    encoder      : fitted OrdinalEncoder (save for inference)
    cat_cols     : list of column names that were encoded
    """
    exclude_cols = exclude_cols or []
    df = df.copy()

    try:
        cat_cols = df.select_dtypes(include='str').columns.difference(exclude_cols).tolist()
    except Exception:
        cat_cols = df.select_dtypes(include='object').columns.difference(exclude_cols).tolist()

    if not cat_cols:
        print("No categorical columns to encode.")
        return df, None, []

    encoder = OrdinalEncoder(
        handle_unknown='use_encoded_value',
        unknown_value=-1,
        dtype=np.int32
    )
    df[cat_cols] = encoder.fit_transform(df[cat_cols])

    return df, encoder, cat_cols


df, cat_encoder, encoded_cols = encode_categoricals(df, exclude_cols=['target'])

print(f"Encoded {len(encoded_cols)} categorical column(s): {encoded_cols}")
print("\nDtypes after encoding:")
print(df.dtypes.value_counts())

In [ ]:
# ============================================================
# Cell 8 — Final Validation
# ============================================================

def validate_pipeline(df: pd.DataFrame) -> None:
    """
    Runs a suite of assertions to confirm the cleaned DataFrame
    is ready for model training.
    """
    print("=" * 50)
    print("PIPELINE VALIDATION REPORT")
    print("=" * 50)

    # 1. Shape
    print(f"\n📐 Shape          : {df.shape}")
    assert df.shape[0] > 0, "DataFrame is empty!"

    # 2. No missing values
    nulls = df.isnull().sum().sum()
    print(f"❓ Missing values  : {nulls}")
    assert nulls == 0, f"Found {nulls} missing values!"

    # 3. Target integrity
    assert 'target' in df.columns, "'target' column missing!"
    assert df['target'].isin([0, 1]).all(), "Target has values outside {0, 1}!"
    dist = df['target'].value_counts()
    pos_rate = df['target'].mean()
    print(f"🎯 Target classes  : {dist.to_dict()}")
    print(f"   Positive rate   : {pos_rate:.2%}")
    assert df['target'].nunique() == 2, "Target must have exactly 2 classes!"

    # 4. No object/string columns remaining
    try:
        obj_cols = df.select_dtypes(include='str').columns.tolist()
    except Exception:
        obj_cols = df.select_dtypes(include='object').columns.tolist()
    print(f"🔤 Remaining str cols: {obj_cols}")
    assert len(obj_cols) == 0, f"Unenoded string columns still present: {obj_cols}"

    # 5. No leakage columns
    leakage_guard = ['readmitted', 'readmit_binary']
    found_leakage = [c for c in leakage_guard if c in df.columns]
    print(f"🚨 Leakage columns : {found_leakage}")
    assert len(found_leakage) == 0, f"Leakage columns still present: {found_leakage}"

    # 6. Duplicates
    dups = df.duplicated().sum()
    print(f"👥 Duplicate rows  : {dups}")

    print("\n" + "=" * 50)
    print("✅ ALL CHECKS PASSED — dataset ready for modelling.")
    print("=" * 50)


validate_pipeline(df)

print("\nSample (first 5 rows):")
df.head()